# ElectricityLoadDiagrams20112014 — Exploration interactive

Objectifs :

- comprendre la structure du dataset ;
- visualiser les profils temporels de consommation ;
- observer les différences entre individus ;
- identifier les saisonnalités journalières, hebdomadaires et annuelles ;
- introduire les notions de **lag**, de **rolling mean** et de **split temporel** ;
- préparer l'analyse de **data drift** entre différentes périodes.

## 1. Installation et imports

Le dataset UCI original est généralement distribué sous forme d'un fichier texte séparé par `;`:

```
"";"MT_001";"MT_002";"MT_003";"MT_004";"MT_005";"MT_006";"MT_007";...
"2011-01-01 00:15:00";0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;...
```

- une colonne temporelle en première position ;
- une colonne par client, nommée `MT_001`, `MT_002`, etc.

In [1]:
# %pip install pandas numpy plotly pyarrow ipywidgets

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

In [2]:
DATA_PATH = Path("/Users/antoinelaborde/workspace/eni/dataset_analysis/data/LD2011_2014.txt")

PosixPath('/Users/antoinelaborde/workspace/eni/dataset_analysis/data/LD2011_2014.txt')

## 2. Chargement des données

In [24]:
df_raw = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=",",
)

date_col = df_raw.columns[0]
df_raw = df_raw.rename(columns={date_col: "DateTime"})
df_raw["DateTime"] = pd.to_datetime(df_raw["DateTime"])

In [25]:
print(f"Nombre de lignes : {len(df_raw):,}")
print(f"Nombre de colonnes : {df_raw.shape[1]:,}")
print(f"Période : {df_raw['DateTime'].min()} -> {df_raw['DateTime'].max()}")
print(f"Nombre de clients : {df_raw.shape[1] - 1}")

Nombre de lignes : 140,256
Nombre de colonnes : 371
Période : 2011-01-01 00:15:00 -> 2015-01-01 00:00:00
Nombre de clients : 370


## 3. Vérifications de qualité des données

Avant de modéliser une série temporelle, on vérifie notamment :

- les valeurs manquantes ;
- les doublons temporels ;
- la fréquence des observations ;
- les valeurs nulles ou négatives.

In [26]:
quality = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(3),
})

quality.head(10)

,dtype,missing,missing_pct
DateTime,datetime64[ns],0,0.0
MT_001,float64,0,0.0
MT_002,float64,0,0.0
MT_003,float64,0,0.0
MT_004,float64,0,0.0
MT_005,float64,0,0.0
MT_006,float64,0,0.0
MT_007,float64,0,0.0
MT_008,float64,0,0.0
MT_009,float64,0,0.0


In [27]:
client_cols = [c for c in df_raw.columns if c != "DateTime"]

negative_values = (df_raw[client_cols] < 0).sum().sum()
zero_ratio = (df_raw[client_cols] == 0).mean().mean()

print("Valeurs négatives :", int(negative_values))
print(f"Part moyenne de zéros : {zero_ratio:.2%}")

Valeurs négatives : 0
Part moyenne de zéros : 20.15%


## 4. Passage au format long

Pour l'analyse et la visualisation, un format long est souvent plus pratique :

| DateTime | client | consumption |
|---|---|---|
| ... | MT_001 | ... |
| ... | MT_002 | ... |

In [28]:
df_long = df_raw.sort_values("DateTime").set_index("DateTime")
df_long.head()

,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
DateTime,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Vue globale de la consommation

On commence par agréger tous les clients afin d'observer la consommation totale du parc.

In [51]:
global_load = df_long.sum(axis=1).rename("total_consumption")

daily_global = global_load.resample("D").sum().reset_index()

fig = px.line(
    daily_global,
    x="DateTime",
    y="total_consumption",
    title="Consommation totale agrégée — évolution quotidienne",
)
fig.update_layout(hovermode="x unified")
fig.show()

## 6. Comparaison des années

Une visualisation par mois permet de comparer les profils saisonniers entre années.

In [30]:
monthly_global = global_load.resample("MS").sum().to_frame()
monthly_global["year"] = monthly_global.index.year
monthly_global["month"] = monthly_global.index.month

fig = px.line(
    monthly_global.reset_index(),
    x="month",
    y="total_consumption",
    color="year",
    markers=True,
    title="Profil mensuel de consommation par année",
)
fig.update_xaxes(dtick=1)
fig.show()

## 7. Profil journalier moyen

Le dataset est une série temporelle haute fréquence. On peut donc calculer un profil moyen selon l'heure de la journée.

In [32]:
mean_load = df_long.mean(axis=1).rename("mean_consumption").to_frame()
mean_load["hour"] = mean_load.index.hour + mean_load.index.minute / 60

hour_profile = (
    mean_load.groupby("hour")["mean_consumption"]
    .mean()
    .reset_index()
)

fig = px.line(
    hour_profile,
    x="hour",
    y="mean_consumption",
    title="Profil journalier moyen",
)
fig.update_xaxes(title="Heure de la journée")
fig.show()

## 8. Profil hebdomadaire

On compare ici les différents jours de la semaine.

In [33]:
weekday_names = {
    0: "Lundi",
    1: "Mardi",
    2: "Mercredi",
    3: "Jeudi",
    4: "Vendredi",
    5: "Samedi",
    6: "Dimanche",
}

tmp = df_long.mean(axis=1).rename("mean_consumption").to_frame()
tmp["weekday"] = tmp.index.dayofweek
tmp["weekday_name"] = tmp["weekday"].map(weekday_names)

weekday_profile = (
    tmp.groupby(["weekday", "weekday_name"])["mean_consumption"]
    .mean()
    .reset_index()
    .sort_values("weekday")
)

fig = px.bar(
    weekday_profile,
    x="weekday_name",
    y="mean_consumption",
    title="Consommation moyenne par jour de la semaine",
)
fig.show()

## 9. Heatmap heure × jour de semaine

Cette représentation met en évidence plusieurs saisonnalités simultanément.

In [34]:
heat = df_long.mean(axis=1).rename("mean_consumption").to_frame()
heat["weekday"] = heat.index.dayofweek
heat["hour"] = heat.index.hour

heat = (
    heat.groupby(["weekday", "hour"])["mean_consumption"]
    .mean()
    .reset_index()
)

pivot = heat.pivot(index="weekday", columns="hour", values="mean_consumption")
pivot.index = [weekday_names[i] for i in pivot.index]

fig = px.imshow(
    pivot,
    aspect="auto",
    labels={"x": "Heure", "y": "Jour", "color": "Consommation moyenne"},
    title="Heatmap — jour de semaine × heure",
)
fig.show()

## 10. Analyse d'un individu

In [35]:
CLIENT = "MT_001"

client_series = df_long[CLIENT]

client_daily = client_series.resample("D").sum().reset_index(name="consumption")

fig = px.line(
    client_daily,
    x="DateTime",
    y="consumption",
    title=f"Consommation quotidienne — {CLIENT}",
)
fig.show()

### Comparaison de plusieurs clients

In [36]:
SELECTED_CLIENTS = ["MT_001", "MT_010", "MT_050", "MT_100"]

existing_clients = [c for c in SELECTED_CLIENTS if c in df_raw.columns]

sample = (
    df_long[existing_clients]
    .resample("D")
    .sum()
    .reset_index()
    .melt(id_vars="DateTime", var_name="client", value_name="consumption")
)

fig = px.line(
    sample,
    x="DateTime",
    y="consumption",
    color="client",
    title="Comparaison de plusieurs profils individuels",
)
fig.show()

## 11. Distribution des consommations par client

Tous les individus ne consomment pas dans les mêmes ordres de grandeur.

In [37]:
client_stats = pd.DataFrame({
    "mean": df_long.mean(),
    "median": df_long.median(),
    "std": df_long.std(),
    "max": df_long.max(),
    "zero_pct": (df_long == 0).mean() * 100,
})

client_stats.describe()

,mean,median,std,max,zero_pct
count,370.000000,370.000000,370.000000,370.000000,370.000000
mean,528.532277,457.574301,316.602729,1750.244236,20.151071
std,2416.149483,1864.856631,2118.138560,11034.768047,22.214166
min,0.818058,0.000000,2.260778,22.613065,0.000000
25%,41.895166,34.100512,29.873039,167.863408,0.008734
50%,106.694350,102.517439,57.757440,315.047798,24.991444
75%,301.967730,283.009215,129.807757,806.285248,25.006595
max,37607.987537,24100.000000,38691.954832,192800.000000,88.637919


In [38]:
fig = px.histogram(
    client_stats.reset_index(),
    x="mean",
    nbins=50,
    title="Distribution de la consommation moyenne des clients",
    labels={"mean": "Consommation moyenne"},
)
fig.show()

## 12. Clients actifs et valeurs nulles

Dans ce dataset, certains clients peuvent présenter de longues périodes à zéro.

Cela peut correspondre à :

- une absence de consommation ;
- une période avant l'entrée du client dans le dataset ;
- une période après sa sortie ;
- une particularité de collecte.

Il ne faut donc pas automatiquement considérer toutes les valeurs nulles comme des anomalies.

In [39]:
zero_pct = ((df_raw == 0).mean() * 100).sort_values(ascending=False)

display(zero_pct.head(15).to_frame("zero_pct"))

,zero_pct
MT_178,88.637919
MT_133,86.812685
MT_181,79.397673
MT_109,78.306097
MT_116,78.305384
MT_112,77.921087
MT_160,77.344285
MT_347,76.591376
MT_337,76.180698
MT_115,75.920460


## 13. Introduction des lags

En séries temporelles, la valeur passée constitue souvent une excellente variable explicative.

In [43]:
feature_df = df_long[[CLIENT]].copy()
feature_df.columns = ["target"]

# La fréquence historique du dataset est de 15 minutes.
# 96 pas = 24 heures.
feature_df["lag_15min"] = feature_df["target"].shift(1)
feature_df["lag_1h"] = feature_df["target"].shift(4)
feature_df["lag_24h"] = feature_df["target"].shift(96)
feature_df["lag_7d"] = feature_df["target"].shift(96 * 7)

feature_df.head(100)

,target,lag_15min,lag_1h,lag_24h,lag_7d
DateTime,,,,,
2011-01-01 00:15:00,0.0,NaN,NaN,NaN,NaN
2011-01-01 00:30:00,0.0,0.0,NaN,NaN,NaN
2011-01-01 00:45:00,0.0,0.0,NaN,NaN,NaN
2011-01-01 01:00:00,0.0,0.0,NaN,NaN,NaN
2011-01-01 01:15:00,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...
2011-01-02 00:00:00,0.0,0.0,0.0,NaN,NaN
2011-01-02 00:15:00,0.0,0.0,0.0,0.0,NaN
2011-01-02 00:30:00,0.0,0.0,0.0,0.0,NaN


In [41]:
corr = feature_df.corr()
corr

,target,lag_15min,lag_1h,lag_24h,lag_7d
target,1.000000,0.915172,0.801439,0.730591,0.591586
lag_15min,0.915172,1.000000,0.824325,0.726176,0.587024
lag_1h,0.801439,0.824325,1.000000,0.702567,0.568713
lag_24h,0.730591,0.726176,0.702567,1.000000,0.598772
lag_7d,0.591586,0.587024,0.568713,0.598772,1.000000


In [42]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    title=f"Corrélation entre consommation actuelle et lags — {CLIENT}",
)
fig.show()

## 14. Rolling mean

Les moyennes glissantes permettent de représenter la tendance locale.

In [44]:
rolling = df_long[[CLIENT]].copy()
rolling["rolling_24h"] = rolling[CLIENT].rolling(96).mean()
rolling["rolling_7d"] = rolling[CLIENT].rolling(96 * 7).mean()

period = rolling.loc["2014-01-01":"2014-01-15"].reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=period["DateTime"],
    y=period[CLIENT],
    name="Consommation",
))
fig.add_trace(go.Scatter(
    x=period["DateTime"],
    y=period["rolling_24h"],
    name="Moyenne glissante 24h",
))
fig.add_trace(go.Scatter(
    x=period["DateTime"],
    y=period["rolling_7d"],
    name="Moyenne glissante 7 jours",
))

fig.update_layout(
    title=f"Consommation et rolling means — {CLIENT}",
    hovermode="x unified",
)
fig.show()

## 15. Pourquoi le split temporel est important

Pour un problème de prévision temporelle, il ne faut pas mélanger arbitrairement passé et futur.

Une séparation logique pourrait être :

- entraînement : 2011–2012 ;
- validation : 2013 ;
- test : 2014.

In [45]:
train = df_long.loc[:"2012-12-31"]
validation = df_long.loc["2013-01-01":"2013-12-31"]
test = df_long.loc["2014-01-01":]

print("Train :", train.index.min(), "->", train.index.max(), len(train))
print("Validation :", validation.index.min(), "->", validation.index.max(), len(validation))
print("Test :", test.index.min(), "->", test.index.max(), len(test))

Train : 2011-01-01 00:15:00 -> 2012-12-31 23:45:00 70175
Validation : 2013-01-01 00:00:00 -> 2013-12-31 23:45:00 35040
Test : 2014-01-01 00:00:00 -> 2015-01-01 00:00:00 35041


## 16. Première observation de drift : 2011 vs 2014

Avant d'utiliser un outil spécialisé comme Evidently, on peut déjà comparer visuellement les distributions.

In [46]:
compare = df_long[[CLIENT]].copy()
compare["year"] = compare.index.year

compare_years = compare[compare["year"].isin([2011, 2014])].reset_index()

fig = px.histogram(
    compare_years,
    x=CLIENT,
    color="year",
    barmode="overlay",
    histnorm="probability density",
    nbins=60,
    opacity=0.6,
    title=f"Comparaison des distributions 2011 vs 2014 — {CLIENT}",
)
fig.show()

In [47]:
year_stats = (
    compare_years.groupby("year")[CLIENT]
    .agg(["mean", "median", "std", "min", "max"])
)

year_stats

,mean,median,std,min,max
year,,,,,
2011,0.000000,0.000000,0.000000,0.0,0.000000
2014,4.058161,2.538071,5.584347,0.0,39.340102


## 17. Comparaison d'une feature dérivée

Dans un pipeline ML réel, le drift peut concerner non seulement la donnée brute mais aussi les **features produites par le feature engineering**.

On construit ici un exemple avec un lag journalier.

In [48]:
derived = df_long[[CLIENT]].copy()
derived["lag_24h"] = derived[CLIENT].shift(96)
derived["year"] = derived.index.year

lag_compare = derived[derived["year"].isin([2011, 2014])].dropna().reset_index()

fig = px.histogram(
    lag_compare,
    x="lag_24h",
    color="year",
    barmode="overlay",
    histnorm="probability density",
    nbins=60,
    opacity=0.6,
    title=f"Distribution de lag_24h — 2011 vs 2014 — {CLIENT}",
)
fig.show()

## 18. Mini-dashboard de synthèse

Les cellules suivantes produisent quatre indicateurs simples sur une période choisie.

In [49]:
START_DATE = "2014-01-01"
END_DATE = "2014-03-31"

period = df_long.loc[START_DATE:END_DATE]

dashboard_kpis = {
    "début": period.index.min(),
    "fin": period.index.max(),
    "nombre_observations": len(period),
    "nombre_clients": period.shape[1],
    "consommation_moyenne": period.mean().mean(),
    "consommation_totale": period.sum().sum(),
}

pd.Series(dashboard_kpis, name="value").to_frame()

,value
début,2014-01-01 00:00:00
fin,2014-03-31 23:45:00
nombre_observations,8640
nombre_clients,370
consommation_moyenne,521.013195
consommation_totale,1665574980.476896


In [50]:
daily = period.sum(axis=1).resample("D").sum().reset_index(name="total")

fig = px.line(
    daily,
    x="DateTime",
    y="total",
    title=f"Dashboard — consommation totale du {START_DATE} au {END_DATE}",
)
fig.update_layout(hovermode="x unified")
fig.show()